# Matplotlib 入门：画图基础与金融图表

本笔记本由文章自动生成，代码和[原文](https://wxrkz123.github.io/blog/Python-Matplotlib/)完全一致。文中的解释、公式、图和每段代码的输出，请对照原文阅读。

## 一、安装、导入与中文显示

In [ ]:
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt    #导入画图模块，习惯上命名为plt

print(matplotlib.__version__)    #查看Matplotlib版本信息

plt.rcParams["font.sans-serif"] = ["Microsoft YaHei", "SimHei", "PingFang SC", "WenQuanYi Zen Hei"]    #中文字体
plt.rcParams["axes.unicode_minus"] = False    #正常显示负号

## 二、第一张图：折线图

先准备数据。和 [pandas 入门](https://wxrkz123.github.io/blog/Python-Pandas/)一样，用 NumPy 模拟三只股票 2023 年的日收盘价：

In [ ]:
rng = np.random.default_rng(2119)    #固定随机种子，保证每次结果相同
dates = pd.bdate_range("2023-01-02", "2023-12-29")    #2023年的工作日
daily_ret = rng.normal([0.0006, 0.0002, 0.0009], [0.018, 0.011, 0.026], size=(len(dates), 3))
prices = pd.DataFrame(100 * np.exp(daily_ret.cumsum(axis=0)),
                      index=dates, columns=["股票A", "股票B", "股票C"]).round(2)

plt.figure(figsize=(8, 4))    #新建一张图，宽8英寸、高4英寸
plt.plot(prices.index, prices["股票A"])    #横轴是日期，纵轴是价格
plt.title("股票A的收盘价")
plt.show()

## 三、Figure 和 Axes：两种写法

Matplotlib 有两种写法，网上的代码两种都有，一开始容易混淆：

（示意代码，不在笔记本中运行）

```python
#写法一：直接用plt画图（简单，适合快速画一张图）
plt.plot(x, y)
plt.title("标题")

#写法二：先创建Figure和Axes，再在Axes上画图（推荐）
fig, ax = plt.subplots()
ax.plot(x, y)
ax.set_title("标题")
```

## 四、常用图形

几种最常用的图形：

（示意代码，不在笔记本中运行）

```python
ax.plot(x, y)    #折线图：价格走势、净值曲线
ax.bar(x, height)    #柱状图：各月收益率、各行业市值
ax.scatter(x, y)    #散点图：两个变量之间的关系，比如风险与收益
ax.hist(x, bins=n)    #直方图：收益率的分布
ax.pie(x, labels=标签)    #饼图：资产配置比例
```

用 `plt.subplots(2, 2)` 一次创建 2 行 2 列共 4 个坐标系，分别画柱状图、散点图、直方图和饼图：

In [ ]:
monthly_ret = prices["股票A"].resample("ME").last().pct_change().dropna() * 100    #月收益率（%）
daily = prices.pct_change().dropna() * 100    #日收益率（%）

fig, axes = plt.subplots(2, 2, figsize=(10, 7))    #axes是一个2×2的数组

axes[0, 0].bar(monthly_ret.index.month, monthly_ret)
axes[0, 0].set_title("柱状图：股票A的月收益率（%）")

axes[0, 1].scatter(daily["股票A"], daily["股票C"], s=8)    #s是点的大小
axes[0, 1].set_title("散点图：股票A与股票C的日收益率")

axes[1, 0].hist(daily["股票C"], bins=30)    #bins是分组的个数
axes[1, 0].set_title("直方图：股票C的日收益率")

axes[1, 1].pie([50, 30, 20], labels=["股票", "债券", "现金"], autopct="%.0f%%")    #autopct显示百分比
axes[1, 1].set_title("饼图：资产配置")

fig.tight_layout()    #自动调整子图间距，避免标题和坐标重叠
plt.show()

### 标题、坐标轴、图例与网格

（示意代码，不在笔记本中运行）

```python
ax.set_title("标题")    #标题
ax.set_xlabel("横轴名称")    #横轴标签
ax.set_ylabel("纵轴名称")    #纵轴标签
ax.set_xlim(左, 右)    #横轴范围
ax.set_ylim(下, 上)    #纵轴范围
ax.legend()    #显示图例（需要画图时设置label）
ax.grid(alpha=0.3)    #显示网格线
ax.axhline(y值)    #画一条水平线
ax.annotate("文字", xy=(x, y), xytext=(x2, y2), arrowprops=dict(arrowstyle="->"))    #带箭头的标注
```

把三只股票的走势画在一张图上，加上这些修饰：

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.5))
styles = {"股票A": ("tab:blue", "-"), "股票B": ("tab:green", "--"), "股票C": ("tab:red", "-")}
for name, (color, ls) in styles.items():
    ax.plot(prices.index, prices[name], color=color, ls=ls, lw=1.3, label=name)

ax.axhline(100, color="grey", lw=0.8, ls=":")    #初始价格100元
top = prices["股票C"].idxmax()    #股票C最高价出现的日期
ax.annotate(f"全年最高 {prices['股票C'].max():.1f}", xy=(top, prices["股票C"].max()),
            xytext=(pd.Timestamp("2023-08-01"), 150), arrowprops=dict(arrowstyle="->"))

ax.set_title("三只股票2023年的走势")
ax.set_xlabel("日期")
ax.set_ylabel("价格（元）")
ax.legend(loc="upper left")    #图例放在左上角
ax.grid(alpha=0.3)
fig.tight_layout()
plt.show()

## 七、保存图片

（示意代码，不在笔记本中运行）

```python
fig.savefig("图片名.png", dpi=300)    #保存为PNG，dpi是分辨率
fig.savefig("图片名.pdf")    #保存为PDF（矢量图，放大不模糊）
fig.savefig("图片名.png", dpi=300, bbox_inches="tight")    #裁掉图片四周的空白
```

## 八、股价、均线与成交量

股票软件里最常见的图：上面是价格和均线，下面是成交量。成交量也用随机数模拟，涨的日子画红色、跌的日子画绿色（A 股的习惯）：

In [ ]:
close = prices["股票A"]
volume = pd.Series(rng.lognormal(mean=12, sigma=0.4, size=len(close)), index=close.index) / 1e4    #成交量（万股）
up = close.diff().fillna(0) >= 0    #当天是否上涨

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(9, 6), sharex=True, height_ratios=[3, 1])
ax1.plot(close.index, close, color="black", lw=1, label="收盘价")
ax1.plot(close.index, close.rolling(20).mean(), color="tab:orange", lw=1.3, label="20日均线")
ax1.plot(close.index, close.rolling(60).mean(), color="tab:purple", lw=1.3, label="60日均线")
ax1.set_title("股票A：收盘价、均线与成交量")
ax1.set_ylabel("价格（元）")
ax1.legend(loc="upper left")
ax1.grid(alpha=0.3)

ax2.bar(close.index, volume, color=np.where(up, "tab:red", "tab:green"), width=1)
ax2.set_ylabel("成交量（万股）")
fig.tight_layout()
plt.show()

## 九、收益率分布与正态分布

[NumPy 入门](https://wxrkz123.github.io/blog/Python-Numpy/)介绍了正态分布的随机抽样。金融中经常要检查收益率是否服从正态分布，最直观的方法就是把收益率的直方图和正态分布的曲线画在一起：

In [ ]:
from scipy import stats

r = daily["股票C"]
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.hist(r, bins=30, density=True, color="tab:blue", alpha=0.6, edgecolor="white", label="实际日收益率")
x = np.linspace(r.min(), r.max(), 200)
ax.plot(x, stats.norm.pdf(x, r.mean(), r.std()), color="tab:red", lw=2, label="相同均值和标准差的正态分布")
var95 = np.percentile(r, 5)    #历史模拟法的95% VaR
ax.axvline(var95, color="black", ls="--", lw=1)
ax.text(var95, ax.get_ylim()[1] * 0.9, f" 5%分位数 {var95:.2f}%", fontsize=9)
ax.set_title("股票C日收益率的分布")
ax.set_xlabel("日收益率（%）")
ax.set_ylabel("密度")
ax.legend()
fig.tight_layout()
plt.show()
print(f"有5%的交易日，股票C的跌幅超过 {-var95:.2f}%")

## 十、投资组合的风险与收益

最后画一张投资组合理论中最经典的图。随机生成 3000 组权重，把资金按不同比例分配到三只股票上，计算每个组合的年化收益率和年化波动率，画成散点图，颜色表示夏普比率：

In [ ]:
ret = prices.pct_change().dropna()
mean_ret = ret.mean() * 252    #年化收益率
cov = ret.cov() * 252    #年化协方差矩阵

weights = rng.dirichlet(np.ones(3), size=3000)    #3000组随机权重，每组加起来等于1
port_ret = weights @ mean_ret.values
port_vol = np.sqrt(np.einsum("ij,jk,ik->i", weights, cov.values, weights))
sharpe = (port_ret - 0.02) / port_vol

fig, ax = plt.subplots(figsize=(8, 5))
sc = ax.scatter(port_vol * 100, port_ret * 100, c=sharpe, cmap="viridis", s=6)
fig.colorbar(sc, ax=ax, label="夏普比率")    #颜色条
best = sharpe.argmax()
ax.scatter(port_vol[best] * 100, port_ret[best] * 100, color="red", marker="*", s=250, label="夏普比率最高的组合")
for name in prices.columns:    #单只股票的位置
    ax.scatter(np.sqrt(cov.loc[name, name]) * 100, mean_ret[name] * 100, color="black", marker="s", s=30)
    ax.annotate(name, (np.sqrt(cov.loc[name, name]) * 100, mean_ret[name] * 100), xytext=(5, -12), textcoords="offset points")
ax.set_xlabel("年化波动率（%）")
ax.set_ylabel("年化收益率（%）")
ax.set_title("3000个随机组合的风险与收益")
ax.legend(loc="upper left")
fig.tight_layout()
plt.show()
print("夏普比率最高的组合：", {name: round(float(w), 2) for name, w in zip(prices.columns, weights[best])})